In [ ]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")

In [ ]:
from helper import create_tokenized_word_list
tokenized = create_tokenized_word_list(dataset)

In [ ]:
from helper import create_tokenized_word_list_for_query
query_tokenized = create_tokenized_word_list_for_query(dataset)
print("Queries tokenized!")

In [ ]:
query_tokenized

In [ ]:
query_text_based = []

for query_list in query_tokenized:
    query_text_based.append(" ".join(query_list))

### Stemming

In [ ]:
from nltk.stem import PorterStemmer
porter_stemmer = PorterStemmer()
stemmed_texts = []

for word_list in tokenized:
    stemmed_words = [porter_stemmer.stem(word) for word in word_list]
    stemmed_texts.append(stemmed_words)

print("texts stemmed!")

In [ ]:
queries_stemmed = []

for word_list in query_tokenized:
    stemmed_query_words = [porter_stemmer.stem(word) for word in word_list]
    queries_stemmed.append(stemmed_query_words)

print("queries stemmed!")

In [ ]:
query_stem_text_based = []

for query_list in queries_stemmed:
    query_stem_text_based.append(" ".join(query_list))

In [ ]:
text_based = []

for sentence in stemmed_texts:
    text_based.append(" ".join(sentence))

In [ ]:
text_based

In [ ]:
len(text_based)

### TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf_vect = TfidfVectorizer()
tf_idf_matrix = tf_idf_vect.fit_transform(text_based)
print("TF-IDF matrix ready!")

In [ ]:
query_vectors = tf_idf_vect.transform(query_text_based)

In [ ]:
query_vectors_stemmed = tf_idf_vect.transform(query_stem_text_based)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(query_vectors, tf_idf_matrix)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarities_queries_stemmed = cosine_similarity(query_vectors_stemmed, tf_idf_matrix)

In [ ]:
similarities

In [ ]:
similarities_queries_stemmed

In [ ]:
from collections import defaultdict
from helper import Scoredoc

doc_dict = defaultdict(str)

for i, doc in enumerate(dataset.docs_iter()):
    doc_dict[i] = doc.doc_id

doc_dict = dict(doc_dict)

qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

score_doc_dict = dict(score_doc_dict)

print("necessary dicts ready!")

In [ ]:
import pandas as pd
query_ids = [query.query_id for query in dataset.queries_iter()]
df = pd.DataFrame(query_ids, columns=["Query_ID"])
df

In [ ]:
import pandas as pd
query_ids = [query.query_id for query in dataset.queries_iter()]
df_new = pd.DataFrame(query_ids, columns=["Query_ID"])
df_new

In [ ]:
from helper import create_AP, create_ndcg, create_statistical_columns
df_new = create_statistical_columns(df_new, qrels_dict, doc_dict, similarities_queries_stemmed)
df_new = create_AP(df_new, qrels_dict, doc_dict, similarities_queries_stemmed)
df_new = create_ndcg(df_new, doc_dict, similarities_queries_stemmed, score_doc_dict)

In [ ]:
df

In [ ]:
df_new

In [ ]:
mydict = {
    "Method": "TF-IDF - Porter's Stemmer",
    "recall_5_mean": df["recall_5"].mean(),
    "recall_5_std": df["recall_5"].std(),
    "recall_5_max": df["recall_5"].max(),
    "recall_5_min": df["recall_5"].min(),
    "recall_10_mean": df["recall_10"].mean(),
    "recall_10_std": df["recall_10"].std(),
    "recall_10_max": df["recall_10"].max(),
    "recall_10_min": df["recall_10"].min(),
    "precision_5_mean": df["precision_5"].mean(),
    "precision_5_std": df["precision_5"].std(),
    "precision_5_max": df["precision_5"].max(),
    "precision_5_min": df["precision_5"].min(),
    "precision_10_mean": df["precision_10"].mean(),
    "precision_10_std": df["precision_10"].std(),
    "precision_10_max": df["precision_10"].max(),
    "precision_10_min": df["precision_10"].min(),
    "f_score_5_mean": df["f_score_5"].mean(),
    "f_score_5_std": df["f_score_5"].std(),
    "f_score_5_max": df["f_score_5"].max(),
    "f_score_5_min": df["f_score_5"].min(),
    "f_score_10_mean": df["f_score_10"].mean(),
    "f_score_10_std": df["f_score_10"].std(),
    "f_score_10_max": df["f_score_10"].max(),
    "f_score_10_min": df["f_score_10"].min(),
    "MAP_5": df["AP_5"].mean(),
    "MAP_10": df["AP_10"].mean(),
    "NDCG_5_mean": df["NDCG_5"].mean(),
    "NDCG_5_std": df["NDCG_5"].std(),
    "NDCG_5_max": df["NDCG_5"].max(),
    "NDCG_5_min": df["NDCG_5"].min(),
    "NDCG_10_mean": df["NDCG_10"].mean(),
    "NDCG_10_std": df["NDCG_10"].std(),
    "NDCG_10_max": df["NDCG_10"].max(),
    "NDCG_10_min": df["NDCG_10"].min()
}

In [ ]:
mydict = {
    "Method": "TF-IDF - Porter's Stemmer Queries Stemmed",
    "recall_5_mean": df_new["recall_5"].mean(),
    "recall_5_std": df_new["recall_5"].std(),
    "recall_5_max": df_new["recall_5"].max(),
    "recall_5_min": df_new["recall_5"].min(),
    "recall_10_mean": df_new["recall_10"].mean(),
    "recall_10_std": df_new["recall_10"].std(),
    "recall_10_max": df_new["recall_10"].max(),
    "recall_10_min": df_new["recall_10"].min(),
    "precision_5_mean": df_new["precision_5"].mean(),
    "precision_5_std": df_new["precision_5"].std(),
    "precision_5_max": df_new["precision_5"].max(),
    "precision_5_min": df_new["precision_5"].min(),
    "precision_10_mean": df_new["precision_10"].mean(),
    "precision_10_std": df_new["precision_10"].std(),
    "precision_10_max": df_new["precision_10"].max(),
    "precision_10_min": df_new["precision_10"].min(),
    "f_score_5_mean": df_new["f_score_5"].mean(),
    "f_score_5_std": df_new["f_score_5"].std(),
    "f_score_5_max": df_new["f_score_5"].max(),
    "f_score_5_min": df_new["f_score_5"].min(),
    "f_score_10_mean": df_new["f_score_10"].mean(),
    "f_score_10_std": df_new["f_score_10"].std(),
    "f_score_10_max": df_new["f_score_10"].max(),
    "f_score_10_min": df_new["f_score_10"].min(),
    "MAP_5": df_new["AP_5"].mean(),
    "MAP_10": df_new["AP_10"].mean(),
    "NDCG_5_mean": df_new["NDCG_5"].mean(),
    "NDCG_5_std": df_new["NDCG_5"].std(),
    "NDCG_5_max": df_new["NDCG_5"].max(),
    "NDCG_5_min": df_new["NDCG_5"].min(),
    "NDCG_10_mean": df_new["NDCG_10"].mean(),
    "NDCG_10_std": df_new["NDCG_10"].std(),
    "NDCG_10_max": df_new["NDCG_10"].max(),
    "NDCG_10_min": df_new["NDCG_10"].min()
}

In [ ]:
df_csv = pd.DataFrame(mydict, index=[0])
df_csv

In [ ]:
df_csv.to_parquet("TF-IDFPorterStemmer.parquet")
print("Parquet dosyası kaydedildi!")

In [ ]:
df_parquet = pd.DataFrame(mydict, index=[0])
df_parquet

In [ ]:
df_parquet.to_parquet("TF-IDFPorterStemmerQueriesStemmed.parquet")
print("parquet")